# Refugio Animal: gestor de animales con Tkinter

## Sobre este cuaderno

Este cuaderno es la **documentación didáctica** del ejercicio:
explica sección a sección el núcleo (diccionario + CRUD + Tkinter + **ficha POO**
+ **especies personalizadas**).

La ventana que abre este notebook es una **demo reducida**.  
El **programa completo de la entrega** está en `tarea1/` (`app.py` / `main.py`).

```bash
cd tarea1
python app.py
# o: python main.py
```

| Capa | Carpeta / módulo | Contenido |
| --- | --- | --- |
| Arranque | `app.py`, `main.py` | Ventana y `mainloop` |
| Datos | `datos/` | Inventario CRUD, ficha clínica, adjuntos, **modelos POO**, **catálogo de especies** |
| Validación | `validaciones/reglas.py` | `validar_datos` (edad 0–30, peso 0.01–100 kg) |
| Interfaz | `interfaz/` | `GestorAnimales`, ficha clínica, *Gestionar especies*, estilos |
| Exportación | `exportacion/pdf_fichas.py` | PDF de fichas |
| Tests / docs | `tests/`, `docs/` | unittest + RF/CU/UML/manuales |


## Cuaderno vs programa (`tarea1/app.py`)

Si ejecutas este notebook y luego `python main.py` dentro de `tarea1/`,
**verás interfaces distintas**. Es intencional.

| Aspecto | Este cuaderno (demo) | `tarea1` (`app.py` / `main.py`) |
| --- | --- | --- |
| Rol | Explicar conceptos paso a paso | Entrega ejecutable |
| Código | Celdas en el `.ipynb` | `interfaz/gestor.py` y módulos |
| CRUD + filtros + tabla | Sí | Sí |
| Panel ficha POO | Sí | Sí |
| Scrollbars en ficha (comportamiento / alimentación / cuidados) | Sí | Sí |
| Especies personalizadas | Demo en celdas (`EspeciePersonalizada`) | *Gestionar especies* + JSON |
| Miniatura de foto | No | Sí |
| Ficha clínica / adjuntos | No | Sí |
| Exportar PDF | No | Sí |
| Estilos (`estilos.py`) | ttk por defecto | Tema tipográfico completo |
| Título | *…(demo cuaderno)* | *Refugio Animal - Gestión de animales* |

Tabla también en `tarea1/README.md`.

Los datos viven **en memoria**; al reiniciar el kernel se pierden los cambios de la sesión
(salvo el JSON de especies personalizadas en `tarea1/datos/`).

## Documentación de `Tema3-InterfacesGraficas` utilizada

| Lección | Aplicación en el proyecto |
| --- | --- |
| [02 - Gestores de geometría](Tema3-InterfacesGraficas/02_GestoresGeometria.ipynb) | `pack()` en bloques; `grid()` dentro de `LabelFrame` |
| [03 - Variables de control](Tema3-InterfacesGraficas/03_VariablesControl.ipynb) | `StringVar` en el formulario |
| [04 - Frame y LabelFrame](Tema3-InterfacesGraficas/04_Contenedor_Frame_Labelframe.ipynb) | Formulario, filtros, tabla y ficha POO |
| [08 - Button](Tema3-InterfacesGraficas/08_Widget_Button.ipynb) / [09 - Entry](Tema3-InterfacesGraficas/09_Widget_Entry.ipynb) | Acciones CRUD y campos |
| [10 - Eventos](Tema3-InterfacesGraficas/10_EventosGenericos.ipynb) | `KeyRelease`, `<<ComboboxSelected>>`, `<<TreeviewSelect>>` |
| [11 - Cajas de diálogo](Tema3-InterfacesGraficas/11_CajasDialogo.ipynb) | `messagebox` |
| [13 - Widgets TTK](Tema3-InterfacesGraficas/13_WidgetsTematicos_TTK.ipynb) | Controles `ttk` |
| [17 - Combobox](Tema3-InterfacesGraficas/17_Widget_Combobox.ipynb) / [18 - Text](Tema3-InterfacesGraficas/18_Widget_Text.ipynb) | Especie, observaciones y ficha con scroll |
| [22 - Treeview](Tema3-InterfacesGraficas/22_Widget_Treeview.ipynb) | Tabla de inventario |

Tema 2 (POO): ver secciones **2b–2d** más abajo y `Tema2-POO/`.


## 1. Datos iniciales y estructura

`animales` es un diccionario cuyas claves son identificadores numéricos únicos
(como una clave primaria). Cada valor es otro diccionario con los campos de la ficha.

- **Edad:** entero entre 0 y 30.
- **Peso:** número finito entre `PESO_MIN_KG` (0.01) y `PESO_MAX_KG` (100).
- **Especies:** las mismas que en `tarea1/datos/constantes.py`.


In [1]:
import math
import tkinter as tk
from abc import ABC, abstractmethod
from functools import wraps
from tkinter import messagebox, ttk


ESPECIES = ("Perro", "Gato", "Conejo", "Loro", "Hámster")
PESO_MIN_KG = 0.01
PESO_MAX_KG = 100.0

animales = {
    1: {
        "nombre": "Luna",
        "especie": "Perro",
        "edad": 4,
        "peso": 18.5,
        "color": "Canela",
        "observaciones": "Sociable; disfruta de los paseos.",
    },
    2: {
        "nombre": "Misi",
        "especie": "Gato",
        "edad": 3,
        "peso": 4.2,
        "color": "Blanco y gris",
        "observaciones": "Tranquilo; busca una familia paciente.",
    },
    3: {
        "nombre": "Kiko",
        "especie": "Loro",
        "edad": 7,
        "peso": 0.6,
        "color": "Verde",
        "observaciones": "Le gusta escuchar música.",
    },
    4: {
        "nombre": "Toby",
        "especie": "Perro",
        "edad": 1,
        "peso": 8.3,
        "color": "Negro",
        "observaciones": "Juguetón y con mucha energía.",
    },
}

print(f"Registros iniciales: {len(animales)}")
print("Especies:", ESPECIES)
print("Primer registro:", animales[1])


Registros iniciales: 4
Especies: ('Perro', 'Gato', 'Conejo', 'Loro', 'Hámster')
Primer registro: {'nombre': 'Luna', 'especie': 'Perro', 'edad': 4, 'peso': 18.5, 'color': 'Canela', 'observaciones': 'Sociable; disfruta de los paseos.'}


## 2. Operaciones CRUD y validación

Las funciones separan la **gestión de datos** de la interfaz. Reciben el
diccionario como argumento (se pueden probar sin GUI). Los errores se
comunican con `ValueError`; la interfaz los muestra con `messagebox`.

Misma idea que en `tarea1/validaciones/reglas.py` + `tarea1/datos/inventario.py`.


In [2]:
def validar_datos(datos):
    """Comprueba los datos del formulario y devuelve un registro limpio."""
    nombre = str(datos.get("nombre", "")).strip()
    especie = str(datos.get("especie", "")).strip()
    color = str(datos.get("color", "")).strip()
    observaciones = str(datos.get("observaciones", "")).strip()
    edad = datos.get("edad")

    if not nombre:
        raise ValueError("El nombre es obligatorio.")
    personalizadas = globals().get("ESPECIES_PERSONALIZADAS", {})
    especies_ok = ESPECIES + tuple(personalizadas.keys())
    if especie not in especies_ok:
        raise ValueError("Selecciona una especie de la lista.")
    if not color:
        raise ValueError("El color es obligatorio.")
    if isinstance(edad, bool) or not isinstance(edad, int):
        raise ValueError("La edad debe ser un número entero.")
    if edad < 0 or edad > 30:
        raise ValueError("La edad debe estar entre 0 y 30 años.")

    try:
        peso = float(datos.get("peso"))
    except (TypeError, ValueError) as error:
        raise ValueError("El peso debe ser un número válido.") from error
    if not math.isfinite(peso):
        raise ValueError("El peso debe ser un número finito.")
    if peso < PESO_MIN_KG or peso > PESO_MAX_KG:
        raise ValueError(
            f"El peso debe estar entre {PESO_MIN_KG} y {PESO_MAX_KG} kg."
        )

    return {
        "nombre": nombre,
        "especie": especie,
        "edad": edad,
        "peso": peso,
        "color": color,
        "observaciones": observaciones,
    }


def crear_animal(datos, inventario):
    """Valida y añade un animal; devuelve el nuevo identificador."""
    registro = validar_datos(datos)
    nuevo_id = max(inventario, default=0) + 1
    inventario[nuevo_id] = registro
    return nuevo_id


def leer_animal(animal_id, inventario):
    """Devuelve una copia de un registro para evitar cambios accidentales."""
    if animal_id not in inventario:
        raise ValueError("No se ha encontrado ese animal.")
    return inventario[animal_id].copy()


def actualizar_animal(animal_id, datos, inventario):
    """Valida y sustituye los datos del animal indicado."""
    if animal_id not in inventario:
        raise ValueError("No se ha encontrado ese animal.")
    inventario[animal_id] = validar_datos(datos)


def borrar_animal(animal_id, inventario):
    """Elimina el animal indicado y devuelve sus datos."""
    if animal_id not in inventario:
        raise ValueError("No se ha encontrado ese animal.")
    return inventario.pop(animal_id)


## 2b. Capa POO (Tema 2): herencia, `@property` y ficha informativa

El enunciado pide **diccionarios** para el CRUD. Además hay una capa POO que
**participa en la interfaz**: al seleccionar un animal se genera una ficha con
los métodos del objeto (polimorfismo).

| Principio | En este cuaderno / `tarea1/` |
| --- | --- |
| **Herencia** | `Perro`, `Gato`, … y `EspeciePersonalizada` ← `AnimalRefugio` |
| **Encapsulación** | `@property` / setters (nombre, edad, peso, color, observaciones) |
| **Abstracción** | Métodos abstractos + `generar_ficha()` en la clase base |
| **Polimorfismo** | Cada especie (nativa o personalizada) responde distinto |

**Recorrido:** `dict` del inventario → `registro_a_animal()` → objeto → `generar_ficha()` → panel Tkinter
(con **scrollbars** en comportamiento, alimentación y cuidados).

Los objetos son **temporales** (se crean al consultar). El inventario sigue siendo un `dict`.


In [3]:
def cuenta_usos(metodo):
    """Decorador propio: cuenta invocaciones en self._usos_<nombre>."""

    @wraps(metodo)
    def envoltura(self, *args, **kwargs):
        atributo = f"_usos_{metodo.__name__}"
        setattr(self, atributo, getattr(self, atributo, 0) + 1)
        return metodo(self, *args, **kwargs)

    return envoltura


class AnimalRefugio(ABC):
    """Contrato común (no se instancia)."""

    ESPECIE = ""

    def __init__(self, nombre, edad, peso, color, observaciones=""):
        self.nombre = nombre
        self.edad = edad
        self.peso = peso
        self.color = color
        self.observaciones = observaciones

    @property
    def nombre(self):
        return self._nombre

    @nombre.setter
    def nombre(self, valor):
        if not isinstance(valor, str) or not valor.strip():
            raise ValueError("El nombre es obligatorio.")
        self._nombre = valor.strip()

    @property
    def edad(self):
        return self._edad

    @edad.setter
    def edad(self, valor):
        if isinstance(valor, bool) or not isinstance(valor, int):
            raise ValueError("La edad debe ser un número entero.")
        if valor < 0 or valor > 30:
            raise ValueError("La edad debe estar entre 0 y 30 años.")
        self._edad = valor

    @property
    def peso(self):
        return self._peso

    @peso.setter
    def peso(self, valor):
        peso = float(valor)
        if not math.isfinite(peso) or peso < PESO_MIN_KG or peso > PESO_MAX_KG:
            raise ValueError(
                f"El peso debe estar entre {PESO_MIN_KG} y {PESO_MAX_KG} kg."
            )
        self._peso = peso

    @property
    def color(self):
        return self._color

    @color.setter
    def color(self, valor):
        if not isinstance(valor, str) or not valor.strip():
            raise ValueError("El color es obligatorio.")
        self._color = valor.strip()

    @property
    def observaciones(self):
        return self._observaciones

    @observaciones.setter
    def observaciones(self, valor):
        self._observaciones = str(valor or "").strip()

    @property
    def especie(self):
        return self.ESPECIE

    @cuenta_usos
    def info_basica(self):
        return f"{self.nombre} ({self.especie}), {self.edad} años, {self.peso:.2f} kg"

    @abstractmethod
    def hacer_sonido(self):
        ...

    @abstractmethod
    def tipo_alimentacion(self):
        ...

    @abstractmethod
    def cuidados_especificos(self):
        """Orientaciones generales (no son consejo veterinario)."""
        ...

    def generar_ficha(self):
        """La base ensambla; las subclases aportan el contenido polimórfico."""
        observaciones = self.observaciones or "(sin observaciones)"
        return {
            "generales": (
                f"Nombre: {self.nombre}\n"
                f"Especie: {self.especie}\n"
                f"Edad: {self.edad} años\n"
                f"Peso: {self.peso:.2f} kg\n"
                f"Color: {self.color}\n"
                f"Observaciones: {observaciones}"
            ),
            "comportamiento": self.hacer_sonido(),
            "alimentacion": self.tipo_alimentacion(),
            "cuidados": self.cuidados_especificos(),
        }


class Perro(AnimalRefugio):
    ESPECIE = "Perro"

    def hacer_sonido(self):
        return f"{self.nombre} dice: ¡Guau!"

    def tipo_alimentacion(self):
        return "omnívoro"

    def cuidados_especificos(self):
        return (
            "Necesita paseos y ejercicio diario, contacto social y un entorno "
            "seguro. Mantén higiene básica y enriquecimiento con juego. Ante "
            "dudas de salud, consulta a un veterinario."
        )


class Gato(AnimalRefugio):
    ESPECIE = "Gato"

    def hacer_sonido(self):
        return f"{self.nombre} dice: ¡Miau!"

    def tipo_alimentacion(self):
        return "carnívoro"

    def cuidados_especificos(self):
        return (
            "Requiere arenero limpio, zonas de descanso y oportunidades de "
            "rascado y juego. Respeta su tranquilidad. Ante síntomas, valora "
            "una consulta veterinaria."
        )


class Conejo(AnimalRefugio):
    ESPECIE = "Conejo"

    def hacer_sonido(self):
        return f"{self.nombre} dice: … (silencio típico de conejo)"

    def tipo_alimentacion(self):
        return "herbívoro"

    def cuidados_especificos(self):
        return (
            "Precisa espacio para moverse, entorno limpio y heno de buena "
            "calidad como base de la dieta. Manipúlalo con calma."
        )


class Loro(AnimalRefugio):
    ESPECIE = "Loro"

    def hacer_sonido(self):
        return f"{self.nombre} dice: ¡Aaah!"

    def tipo_alimentacion(self):
        return "granívoro"

    def cuidados_especificos(self):
        return (
            "Necesita interacción social, estímulos mentales y un alojamiento "
            "amplio con perchas y juguetes seguros."
        )


class Hamster(AnimalRefugio):
    ESPECIE = "Hámster"

    def hacer_sonido(self):
        return f"{self.nombre} dice: ¡Iiik!"

    def tipo_alimentacion(self):
        return "omnívoro"

    def cuidados_especificos(self):
        return (
            "Es mayoritariamente nocturno: ofrece jaula segura, rueda adecuada "
            "y un entorno tranquilo de día."
        )



class EspeciePersonalizada(AnimalRefugio):
    """Especie definida por configuración (sin clase Python nueva por cada alta)."""

    ESPECIE = ""

    def __init__(
        self,
        nombre,
        edad,
        peso,
        color,
        observaciones="",
        *,
        nombre_especie,
        sonido,
        alimentacion,
        cuidados,
    ):
        self._nombre_especie = str(nombre_especie).strip()
        self._sonido = str(sonido).strip()
        self._alimentacion = str(alimentacion).strip()
        self._cuidados = str(cuidados).strip()
        super().__init__(nombre, edad, peso, color, observaciones)

    @property
    def especie(self):
        return self._nombre_especie

    def hacer_sonido(self):
        return f"{self.nombre}: {self._sonido}"

    def tipo_alimentacion(self):
        return self._alimentacion

    def cuidados_especificos(self):
        return self._cuidados


CLASES = {
    "Perro": Perro,
    "Gato": Gato,
    "Conejo": Conejo,
    "Loro": Loro,
    "Hámster": Hamster,
}


# Configuración en memoria de especies personalizadas (en tarea1/ va a JSON)
ESPECIES_PERSONALIZADAS = {}


def registro_a_animal(registro):
    """Adaptador dict → objeto (equivalente a datos/adaptador_poo.py)."""
    especie = registro["especie"]
    clase = CLASES.get(especie)
    if clase is not None:
        return clase(
            registro["nombre"],
            registro["edad"],
            registro["peso"],
            registro["color"],
            registro.get("observaciones", ""),
        )
    cfg = ESPECIES_PERSONALIZADAS.get(especie)
    if cfg is None:
        raise ValueError(f"Demo POO: sin clase ni config para {especie}")
    return EspeciePersonalizada(
        registro["nombre"],
        registro["edad"],
        registro["peso"],
        registro["color"],
        registro.get("observaciones", ""),
        nombre_especie=cfg["nombre"],
        sonido=cfg["sonido"],
        alimentacion=cfg["alimentacion"],
        cuidados=cfg["cuidados"],
    )


# Demostración: encapsulación + polimorfismo + decorador
demo = registro_a_animal(animales[1])  # Luna (Perro)
print(demo.info_basica())
print(demo.hacer_sonido(), "|", demo.tipo_alimentacion())
print("Cuidados:", demo.cuidados_especificos()[:80], "...")
print("Usos de info_basica:", demo._usos_info_basica)
try:
    demo.peso = 150  # fuera de rango → ValueError del setter
except ValueError as error:
    print("Setter peso:", error)


Luna (Perro), 4 años, 18.50 kg
Luna dice: ¡Guau! | omnívoro
Cuidados: Necesita paseos y ejercicio diario, contacto social y un entorno seguro. Mantén  ...
Usos de info_basica: 1
Setter peso: El peso debe estar entre 0.01 y 100.0 kg.


## 2c. Aplicación práctica: `generar_ficha()` y polimorfismo

La clase base construye la ficha; cada especie aporta sonido, alimentación y
cuidados. **No hace falta** un `if especie == ...` en quien consume la ficha.

Misma llamada (`animal.generar_ficha()`), distintos resultados.


In [4]:
def mostrar_ficha(titulo, ficha):
    print("=" * 60)
    print(titulo)
    print("- Datos generales -")
    print(ficha["generales"])
    print("- Comportamiento -")
    print(ficha["comportamiento"])
    print("- Alimentación -")
    print(ficha["alimentacion"])
    print("- Cuidados específicos -")
    print(ficha["cuidados"])
    print()


# Misma interfaz polimórfica para varias especies del inventario
for animal_id in (1, 2, 3):  # Luna (Perro), Misi (Gato), Kiko (Loro)
    animal = registro_a_animal(animales[animal_id])
    mostrar_ficha(
        f"ID {animal_id} → objeto {type(animal).__name__}",
        animal.generar_ficha(),
    )

# También funciona crear objetos directos (Conejo / Hámster)
ejemplos_extra = [
    Conejo("Nube", 2, 1.5, "Blanco"),
    Hamster("Pip", 1, 0.12, "Marrón"),
]
for animal in ejemplos_extra:
    mostrar_ficha(f"Objeto directo {type(animal).__name__}", animal.generar_ficha())



ID 1 → objeto Perro
- Datos generales -
Nombre: Luna
Especie: Perro
Edad: 4 años
Peso: 18.50 kg
Color: Canela
Observaciones: Sociable; disfruta de los paseos.
- Comportamiento -
Luna dice: ¡Guau!
- Alimentación -
omnívoro
- Cuidados específicos -
Necesita paseos y ejercicio diario, contacto social y un entorno seguro. Mantén higiene básica y enriquecimiento con juego. Ante dudas de salud, consulta a un veterinario.

ID 2 → objeto Gato
- Datos generales -
Nombre: Misi
Especie: Gato
Edad: 3 años
Peso: 4.20 kg
Color: Blanco y gris
Observaciones: Tranquilo; busca una familia paciente.
- Comportamiento -
Misi dice: ¡Miau!
- Alimentación -
carnívoro
- Cuidados específicos -
Requiere arenero limpio, zonas de descanso y oportunidades de rascado y juego. Respeta su tranquilidad. Ante síntomas, valora una consulta veterinaria.

ID 3 → objeto Loro
- Datos generales -
Nombre: Kiko
Especie: Loro
Edad: 7 años
Peso: 0.60 kg
Color: Verde
Observaciones: Le gusta escuchar música.
- Comportamiento -
Kiko

## 2d. Especies personalizadas (`EspeciePersonalizada`)

Las **nativas** (Perro, Gato, …) tienen clase propia. Las **personalizadas**
(Hurón, Tortuga, …) usan **una sola clase** con sonido, alimentación y cuidados
configurados: no hace falta escribir una clase Python nueva ni usar `eval()`.

En `tarea1/` se gestionan desde la GUI (*Gestionar especies*) y se guardan en
`datos/especies_personalizadas.json`. Aquí demostramos el mismo modelo en memoria.


In [5]:
# Alta didáctica de una especie personalizada (en tarea1: JSON + diálogo Tkinter)
ESPECIES_PERSONALIZADAS["Hurón"] = {
    "nombre": "Hurón",
    "sonido": "Emite pequeños sonidos y chillidos",
    "alimentacion": "Carnívoro",
    "cuidados": (
        "Necesita un entorno seguro, enriquecimiento ambiental y supervisión. "
        "Ofrece agua fresca y un alojamiento amplio. Ante dudas de salud, "
        "consulta a un veterinario especializado."
    ),
}

# Animal de inventario "virtual" → adaptador → ficha (igual que en la GUI)
registro_nico = {
    "nombre": "Nico",
    "especie": "Hurón",
    "edad": 2,
    "peso": 1.2,
    "color": "Marrón",
    "observaciones": "Curioso y juguetón",
}
nico = registro_a_animal(registro_nico)
print(type(nico).__name__, "→", nico.especie)
mostrar_ficha("Ficha POO de Nico (Hurón)", nico.generar_ficha())



EspeciePersonalizada → Hurón
Ficha POO de Nico (Hurón)
- Datos generales -
Nombre: Nico
Especie: Hurón
Edad: 2 años
Peso: 1.20 kg
Color: Marrón
Observaciones: Curioso y juguetón
- Comportamiento -
Nico: Emite pequeños sonidos y chillidos
- Alimentación -
Carnívoro
- Cuidados específicos -
Necesita un entorno seguro, enriquecimiento ambiental y supervisión. Ofrece agua fresca y un alojamiento amplio. Ante dudas de salud, consulta a un veterinario especializado.



## 3. Prueba rápida de las cuatro operaciones

Diccionario vacío independiente para no alterar los registros iniciales.


In [6]:
inventario_prueba = {}
datos_prueba = {
    "nombre": "Nube",
    "especie": "Gato",
    "edad": 2,
    "peso": 3.8,
    "color": "Gris",
    "observaciones": "Ejemplo temporal",
}

id_prueba = crear_animal(datos_prueba, inventario_prueba)  # Crear
print("Crear:", id_prueba, inventario_prueba)
print("Leer:", leer_animal(id_prueba, inventario_prueba))

datos_prueba["edad"] = 3
actualizar_animal(id_prueba, datos_prueba, inventario_prueba)  # Actualizar
print("Actualizar:", leer_animal(id_prueba, inventario_prueba))

borrar_animal(id_prueba, inventario_prueba)  # Borrar
print("Borrar: quedan", len(inventario_prueba), "registros.")

# La validación de peso también rechaza valores fuera de rango:
try:
    validar_datos({**datos_prueba, "peso": 150})
except ValueError as error:
    print("Peso inválido:", error)


Crear: 1 {1: {'nombre': 'Nube', 'especie': 'Gato', 'edad': 2, 'peso': 3.8, 'color': 'Gris', 'observaciones': 'Ejemplo temporal'}}
Leer: {'nombre': 'Nube', 'especie': 'Gato', 'edad': 2, 'peso': 3.8, 'color': 'Gris', 'observaciones': 'Ejemplo temporal'}
Actualizar: {'nombre': 'Nube', 'especie': 'Gato', 'edad': 3, 'peso': 3.8, 'color': 'Gris', 'observaciones': 'Ejemplo temporal'}
Borrar: quedan 0 registros.
Peso inválido: El peso debe estar entre 0.01 y 100.0 kg.


## 4. Interfaz gráfica (demo del cuaderno)

GUI **didáctica**: CRUD + filtros + `Treeview` + panel ficha POO con
**scrollbars** en *Comportamiento*, *Alimentación* y *Cuidados específicos*.

**No es la misma ventana que `tarea1/app.py`.** Allí además hay foto, clínica,
PDF, estilos y el diálogo completo *Gestionar especies*.

Al seleccionar una fila:

1. Se rellena el formulario (`dict`).
2. `registro_a_animal(registro)` crea el objeto (nativo o `EspeciePersonalizada`).
3. `generar_ficha()` rellena el panel (sin `if` por especie).


In [7]:
class GestorAnimales:
    """Interfaz Tkinter: CRUD + panel de ficha POO (con scrollbars)."""

    def __init__(self, raiz, inventario):
        self.raiz = raiz
        self.inventario = inventario
        self.nombre_var = tk.StringVar()
        self.especie_var = tk.StringVar(value=ESPECIES[0])
        self.edad_var = tk.StringVar()
        self.peso_var = tk.StringVar()
        self.color_var = tk.StringVar()
        self.busqueda_var = tk.StringVar()
        self.filtro_var = tk.StringVar(value="Todas")
        self.estado_var = tk.StringVar(
            value="Selecciona un animal o completa el formulario."
        )
        self._mensaje_ficha_vacia = (
            "Selecciona un animal de la tabla para consultar su ficha."
        )

        self.raiz.title("Refugio Animal - Gestión de animales (demo cuaderno)")
        self.raiz.geometry("1000x740")
        self.raiz.minsize(860, 660)
        self._crear_controles()
        self._actualizar_tabla()
        self._mostrar_ficha_vacia()

    def _crear_controles(self):
        contenedor = ttk.Frame(self.raiz, padding=12)
        contenedor.pack(fill="both", expand=True)

        ttk.Label(contenedor, text="Refugio Animal").pack(anchor="w")
        ttk.Label(
            contenedor,
            text="Demo del cuaderno — dict + CRUD + ficha POO (completo en tarea1/)",
        ).pack(anchor="w", pady=(0, 10))

        formulario = ttk.LabelFrame(contenedor, text="Datos del animal", padding=10)
        formulario.pack(fill="x", pady=(0, 8))
        formulario.columnconfigure(1, weight=1)
        formulario.columnconfigure(3, weight=1)

        ttk.Label(formulario, text="Nombre:").grid(row=0, column=0, sticky="w", padx=5, pady=4)
        ttk.Entry(formulario, textvariable=self.nombre_var).grid(
            row=0, column=1, sticky="ew", padx=5, pady=4
        )
        ttk.Label(formulario, text="Especie:").grid(row=0, column=2, sticky="w", padx=5, pady=4)
        valores_especie = ESPECIES + tuple(ESPECIES_PERSONALIZADAS.keys())
        self.combo_especie = ttk.Combobox(
            formulario,
            textvariable=self.especie_var,
            values=valores_especie,
            state="readonly",
        )
        self.combo_especie.grid(row=0, column=3, sticky="ew", padx=5, pady=4)

        ttk.Label(formulario, text="Edad (0-30):").grid(row=1, column=0, sticky="w", padx=5, pady=4)
        ttk.Entry(formulario, textvariable=self.edad_var).grid(
            row=1, column=1, sticky="ew", padx=5, pady=4
        )
        ttk.Label(formulario, text="Peso (0.01-100 kg):").grid(
            row=1, column=2, sticky="w", padx=5, pady=4
        )
        ttk.Entry(formulario, textvariable=self.peso_var).grid(
            row=1, column=3, sticky="ew", padx=5, pady=4
        )

        ttk.Label(formulario, text="Color:").grid(row=2, column=0, sticky="w", padx=5, pady=4)
        ttk.Entry(formulario, textvariable=self.color_var).grid(
            row=2, column=1, sticky="ew", padx=5, pady=4
        )
        ttk.Label(formulario, text="Observaciones:").grid(
            row=2, column=2, sticky="nw", padx=5, pady=4
        )
        self.observaciones_text = tk.Text(formulario, height=3, width=35, wrap="word")
        self.observaciones_text.grid(row=2, column=3, sticky="ew", padx=5, pady=4)

        botones = ttk.Frame(formulario)
        botones.grid(row=3, column=0, columnspan=4, sticky="w", pady=(8, 0))
        ttk.Button(botones, text="Añadir", command=self._crear).pack(side="left", padx=3)
        ttk.Button(botones, text="Actualizar", command=self._actualizar).pack(side="left", padx=3)
        ttk.Button(botones, text="Borrar", command=self._borrar).pack(side="left", padx=3)
        ttk.Button(botones, text="Limpiar", command=self._limpiar).pack(side="left", padx=3)

        filtros = ttk.LabelFrame(contenedor, text="Buscar y filtrar", padding=8)
        filtros.pack(fill="x", pady=(0, 8))
        ttk.Label(filtros, text="Nombre:").pack(side="left", padx=(0, 5))
        busqueda = ttk.Entry(filtros, textvariable=self.busqueda_var, width=28)
        busqueda.pack(side="left", padx=(0, 14))
        busqueda.bind("<KeyRelease>", lambda evento: self._actualizar_tabla())
        ttk.Label(filtros, text="Especie:").pack(side="left", padx=(0, 5))
        filtro = ttk.Combobox(
            filtros,
            textvariable=self.filtro_var,
            values=("Todas",) + ESPECIES + tuple(ESPECIES_PERSONALIZADAS.keys()),
            state="readonly",
            width=14,
        )
        filtro.pack(side="left")
        filtro.bind("<<ComboboxSelected>>", lambda evento: self._actualizar_tabla())

        ficha = ttk.LabelFrame(
            contenedor,
            text="Información del animal seleccionado",
            padding=8,
        )
        ficha.pack(fill="x", pady=(0, 8))
        self.ficha_aviso_var = tk.StringVar(value=self._mensaje_ficha_vacia)
        self.lbl_ficha_aviso = ttk.Label(
            ficha, textvariable=self.ficha_aviso_var, wraplength=900
        )
        self.lbl_ficha_aviso.pack(anchor="w", fill="x")

        self.ficha_secciones = ttk.Frame(ficha)
        self.ficha_secciones.columnconfigure(0, weight=1)
        self.ficha_secciones.columnconfigure(1, weight=1)

        def _bloque_etiqueta(padre, titulo, fila, columna):
            marco = ttk.LabelFrame(padre, text=titulo, padding=6)
            marco.grid(row=fila, column=columna, sticky="nsew", padx=4, pady=4)
            etiqueta = ttk.Label(marco, text="", wraplength=420, justify="left")
            etiqueta.pack(anchor="nw", fill="both", expand=True)
            return etiqueta

        def _bloque_scroll(padre, titulo, fila, columna, alto=5):
            marco = ttk.LabelFrame(padre, text=titulo, padding=6)
            marco.grid(row=fila, column=columna, sticky="nsew", padx=4, pady=4)
            marco.columnconfigure(0, weight=1)
            marco.rowconfigure(0, weight=1)
            texto = tk.Text(marco, height=alto, wrap="word", relief="flat", borderwidth=0)
            barra = ttk.Scrollbar(marco, orient="vertical", command=texto.yview)
            texto.configure(yscrollcommand=barra.set, state="disabled")
            texto.grid(row=0, column=0, sticky="nsew")
            barra.grid(row=0, column=1, sticky="ns")
            return texto

        self.lbl_ficha_generales = _bloque_etiqueta(
            self.ficha_secciones, "Datos generales", 0, 0
        )
        self.txt_ficha_comportamiento = _bloque_scroll(
            self.ficha_secciones, "Comportamiento", 0, 1, alto=4
        )
        self.txt_ficha_alimentacion = _bloque_scroll(
            self.ficha_secciones, "Alimentación", 1, 0, alto=4
        )
        self.txt_ficha_cuidados = _bloque_scroll(
            self.ficha_secciones, "Cuidados específicos", 1, 1, alto=6
        )

        tabla_marco = ttk.Frame(contenedor)
        tabla_marco.pack(fill="both", expand=True)
        columnas = ("id", "nombre", "especie", "edad", "peso", "color", "observaciones")
        self.tabla = ttk.Treeview(
            tabla_marco,
            columns=columnas,
            show="headings",
            selectmode="browse",
            height=8,
        )
        encabezados = {
            "id": "ID",
            "nombre": "Nombre",
            "especie": "Especie",
            "edad": "Edad",
            "peso": "Peso (kg)",
            "color": "Color",
            "observaciones": "Observaciones",
        }
        anchos = {
            "id": 45,
            "nombre": 110,
            "especie": 85,
            "edad": 55,
            "peso": 75,
            "color": 100,
            "observaciones": 300,
        }
        for columna in columnas:
            self.tabla.heading(columna, text=encabezados[columna])
            self.tabla.column(columna, width=anchos[columna], anchor="w")
        barra_vertical = ttk.Scrollbar(tabla_marco, orient="vertical", command=self.tabla.yview)
        barra_horizontal = ttk.Scrollbar(
            tabla_marco, orient="horizontal", command=self.tabla.xview
        )
        self.tabla.configure(
            yscrollcommand=barra_vertical.set,
            xscrollcommand=barra_horizontal.set,
        )
        self.tabla.grid(row=0, column=0, sticky="nsew")
        barra_vertical.grid(row=0, column=1, sticky="ns")
        barra_horizontal.grid(row=1, column=0, sticky="ew")
        tabla_marco.rowconfigure(0, weight=1)
        tabla_marco.columnconfigure(0, weight=1)
        self.tabla.bind("<<TreeviewSelect>>", self._seleccionar)

        ttk.Label(contenedor, textvariable=self.estado_var).pack(anchor="w", pady=(8, 0))

    def _datos_del_formulario(self):
        try:
            edad = int(self.edad_var.get().strip())
        except ValueError as error:
            raise ValueError("La edad debe escribirse como un número entero.") from error
        try:
            peso = float(self.peso_var.get().strip().replace(",", "."))
        except ValueError as error:
            raise ValueError(
                "El peso debe escribirse como un número, por ejemplo 4.5."
            ) from error
        return {
            "nombre": self.nombre_var.get(),
            "especie": self.especie_var.get(),
            "edad": edad,
            "peso": peso,
            "color": self.color_var.get(),
            "observaciones": self.observaciones_text.get("1.0", "end"),
        }

    def _id_seleccionado(self):
        seleccion = self.tabla.selection()
        if not seleccion:
            messagebox.showwarning(
                "Falta seleccionar", "Selecciona un animal de la tabla."
            )
            return None
        return int(seleccion[0])

    def _crear(self):
        try:
            animal_id = crear_animal(self._datos_del_formulario(), self.inventario)
        except ValueError as error:
            messagebox.showerror("No se pudo crear el registro", str(error))
            return
        self._actualizar_tabla()
        self._limpiar()
        self.estado_var.set(f"Animal creado con el identificador {animal_id}.")

    def _actualizar(self):
        animal_id = self._id_seleccionado()
        if animal_id is None:
            return
        try:
            actualizar_animal(
                animal_id, self._datos_del_formulario(), self.inventario
            )
        except ValueError as error:
            messagebox.showerror("No se pudo actualizar el registro", str(error))
            return
        self._actualizar_tabla()
        self.estado_var.set(f"Registro {animal_id} actualizado.")

    def _borrar(self):
        animal_id = self._id_seleccionado()
        if animal_id is None:
            return
        nombre = self.inventario[animal_id]["nombre"]
        if not messagebox.askyesno("Confirmar borrado", f"¿Quieres borrar a {nombre}?"):
            return
        try:
            borrar_animal(animal_id, self.inventario)
        except ValueError as error:
            messagebox.showerror("No se pudo borrar el registro", str(error))
            return
        self._actualizar_tabla()
        self._limpiar()
        self.estado_var.set(f"Registro {animal_id} eliminado.")

    @staticmethod
    def _poner_texto_ficha(widget, contenido):
        if isinstance(widget, tk.Text):
            widget.configure(state="normal")
            widget.delete("1.0", "end")
            if contenido:
                widget.insert("1.0", contenido)
            widget.configure(state="disabled")
        else:
            widget.configure(text=contenido)

    def _mostrar_ficha_vacia(self):
        self.ficha_aviso_var.set(self._mensaje_ficha_vacia)
        self.lbl_ficha_aviso.pack(anchor="w", fill="x")
        self.ficha_secciones.pack_forget()
        self._poner_texto_ficha(self.lbl_ficha_generales, "")
        self._poner_texto_ficha(self.txt_ficha_comportamiento, "")
        self._poner_texto_ficha(self.txt_ficha_alimentacion, "")
        self._poner_texto_ficha(self.txt_ficha_cuidados, "")

    def _actualizar_ficha_poo(self, registro):
        try:
            animal = registro_a_animal(registro)
            ficha = animal.generar_ficha()
        except ValueError as error:
            self.ficha_aviso_var.set(
                f"No se pudo generar la ficha POO de este registro: {error}"
            )
            self.lbl_ficha_aviso.pack(anchor="w", fill="x")
            self.ficha_secciones.pack_forget()
            return

        self.lbl_ficha_aviso.pack_forget()
        self.ficha_secciones.pack(fill="both", expand=True)
        self._poner_texto_ficha(self.lbl_ficha_generales, ficha["generales"])
        self._poner_texto_ficha(self.txt_ficha_comportamiento, ficha["comportamiento"])
        self._poner_texto_ficha(self.txt_ficha_alimentacion, ficha["alimentacion"])
        self._poner_texto_ficha(self.txt_ficha_cuidados, ficha["cuidados"])

    def _seleccionar(self, evento=None):
        seleccion = self.tabla.selection()
        if not seleccion:
            self._mostrar_ficha_vacia()
            return
        animal_id = int(seleccion[0])
        registro = leer_animal(animal_id, self.inventario)
        self.nombre_var.set(registro["nombre"])
        self.especie_var.set(registro["especie"])
        self.edad_var.set(str(registro["edad"]))
        self.peso_var.set(str(registro["peso"]))
        self.color_var.set(registro["color"])
        self.observaciones_text.delete("1.0", "end")
        self.observaciones_text.insert("1.0", registro["observaciones"])
        self._actualizar_ficha_poo(registro)
        try:
            animal = registro_a_animal(registro)
            extra = f" | {animal.hacer_sonido()} | {animal.tipo_alimentacion()}"
        except ValueError:
            extra = ""
        self.estado_var.set(f"Registro {animal_id} seleccionado{extra}")

    def _limpiar(self):
        self.nombre_var.set("")
        self.especie_var.set(ESPECIES[0])
        self.edad_var.set("")
        self.peso_var.set("")
        self.color_var.set("")
        self.observaciones_text.delete("1.0", "end")
        seleccion = self.tabla.selection()
        if seleccion:
            self.tabla.selection_remove(seleccion)
        self._mostrar_ficha_vacia()

    def _actualizar_tabla(self):
        for fila in self.tabla.get_children():
            self.tabla.delete(fila)

        texto = self.busqueda_var.get().strip().casefold()
        especie = self.filtro_var.get()
        visibles = 0
        for animal_id, registro in self.inventario.items():
            coincide_nombre = texto in registro["nombre"].casefold()
            coincide_especie = especie == "Todas" or registro["especie"] == especie
            if coincide_nombre and coincide_especie:
                self.tabla.insert(
                    "",
                    "end",
                    iid=str(animal_id),
                    values=(
                        animal_id,
                        registro["nombre"],
                        registro["especie"],
                        registro["edad"],
                        f"{registro['peso']:.2f}",
                        registro["color"],
                        registro["observaciones"],
                    ),
                )
                visibles += 1
        self.estado_var.set(
            f"Registros totales: {len(self.inventario)} | visibles: {visibles}"
        )


## 5. Ejecutar la demo del cuaderno

Orden: **Reiniciar kernel → Ejecutar todas**. Se abre la ventana *demo cuaderno*.

Al seleccionar filas verás el panel **Información del animal seleccionado**
(con barras de desplazamiento en textos largos).

Para la **interfaz completa** (foto, clínica, PDF, *Gestionar especies*):

```bash
cd tarea1
python app.py
# equivalente: python main.py
python ejecutar_tests.py
```


In [8]:
raiz = tk.Tk()
aplicacion = GestorAnimales(raiz, animales)
raiz.mainloop()
